# Pneumothorax Detection - Version 2 Training Pipeline

This notebook provides an easy-to-use interface for training the improved pneumothorax detection models.

## What's New in Version 2:
1. **Attention U-Net++** - Attention gates for better feature focusing
2. **EfficientNet-B3 Classifier** - Better backbone for classification
3. **Focal Loss** - Better handling of class imbalance
4. **Test-Time Augmentation** - Improved inference accuracy
5. **Lower Threshold (0.3)** - Higher recall for safety-critical application

## Training Steps:
1. Set your file paths and parameters
2. Train the EfficientNet classifier (5-10 epochs)
3. Train the Attention U-Net++ segmentation model (80 epochs)
4. Evaluate performance

**Important:** Make sure your data is in DICOM format with proper CSV labels!

## 1. Setup and Configuration

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tensorflow.keras import callbacks
from tensorflow.keras import optimizers

import utils
from callbacks import LearningRateFinder
from generators import SegGenerator, ClassifierGenerator
from losses import (
    weighted_pixel_bce_loss, 
    dice_loss, 
    combined_dice_wpce_loss,
    focal_loss,
    focal_tversky_loss,
    combined_dice_focal_loss
)
from metrics import dice_coefficient_wrapper
from models import create_segmentation_model, create_classification_model

print("✓ All modules imported successfully!")

ModuleNotFoundError: No module named 'imgaug'

## 2. Configure File Paths

**IMPORTANT:** Update these paths to match your system!

In [ ]:
# =================== ADJUST THESE PATHS ===================

# Where to save the trained classifier model
CLASSIFIER_SAVE_PATH = 'C:/Users/sriva/pneumothorax-detection-unet/saved_models/classifier_efficientnet_v2/'

# Where to save the trained segmentation model
SEG_SAVE_PATH = 'C:/Users/sriva/pneumothorax-detection-unet/saved_models/segmentation_attention_unet_v2/'

# Where your DICOM images are stored
IMAGE_PATH = 'C:/Users/sriva/PNEUMOTHORAX/full_short/'

# CSV files with labels
CLASSIFIER_CSV_PATH = 'fin.csv'  # For classifier: ImageId, Class columns
SEGMENTATION_CSV_PATH = 'fin.csv'  # For segmentation: ImageId, EncodedPixels columns

# ==========================================================

## 3. Training Hyperparameters

In [ ]:
# Read from config.ini
BATCH_SIZE, RESIZE_TO, TRAIN_PROP = utils.read_config_file()

# Classifier hyperparameters
CLASSIFIER_EPOCHS = 10  # Increased from 5 for better convergence
CLASSIFIER_LR = 1e-4
CLASSIFIER_BACKBONE = 'EfficientNetB3'  # New! Better than DenseNet

# Segmentation hyperparameters
SEG_EPOCHS = 80
SEG_LR = 3e-4
SEG_ARCHITECTURE = 'attention_unet_plus_plus'  # New! Attention-enhanced U-Net++
SEG_DEPTH = 3  # U-Net++ depth
USE_ATTENTION = True  # Enable attention gates

# Loss function choice for segmentation
LOSS_TYPE = 'combined_dice_focal'  # Options: 'combined_dice_wpce', 'combined_dice_focal', 'focal_tversky'

# Beta pixel weighting (average % of pneumothorax pixels in training set)
BETA_PIXEL_WEIGHTING = 0.010753784

# Classifier threshold (lower = higher recall)
CLASSIFIER_THRESHOLD = 0.3  # New! Lowered from 0.5 for safety

print(f"Batch Size: {BATCH_SIZE}")
print(f"Image Resize: {RESIZE_TO}x{RESIZE_TO}")
print(f"Train/Val Split: {TRAIN_PROP}/{1-TRAIN_PROP}")
print(f"Classifier Backbone: {CLASSIFIER_BACKBONE}")
print(f"Segmentation: {SEG_ARCHITECTURE} (L={SEG_DEPTH}, Attention={USE_ATTENTION})")
print(f"Classification Threshold: {CLASSIFIER_THRESHOLD} (optimized for high recall)")

## 4. Train Classifier Model

This will train the **EfficientNet-B3** classifier to distinguish pneumothorax vs non-pneumothorax X-rays.

In [ ]:
print("\n" + "="*60)
print("TRAINING CLASSIFIER (EfficientNet-B3)")
print("="*60 + "\n")

# Load data
overall_df = pd.read_csv(CLASSIFIER_CSV_PATH, index_col='ImageId')
overall_df_size = len(overall_df)
train_num = int(overall_df_size * TRAIN_PROP)

print(f'Training samples: {train_num}')
print(f'Validation samples: {overall_df_size - train_num}\n')

TRAIN_STEPS = train_num // BATCH_SIZE
VAL_STEPS = (overall_df_size - train_num) // BATCH_SIZE

# Create data generators
train_df = overall_df[:train_num]
val_df = overall_df[train_num:]

train_generator = ClassifierGenerator(train_df, IMAGE_PATH, BATCH_SIZE, resize_to=RESIZE_TO)
val_generator = ClassifierGenerator(val_df, IMAGE_PATH, BATCH_SIZE, resize_to=RESIZE_TO, aug=False)

# Create model
print(f"Creating {CLASSIFIER_BACKBONE} classifier model...")
classifier_model = create_classification_model(RESIZE_TO, bb=CLASSIFIER_BACKBONE)

# Compile model
opt = optimizers.Adam(learning_rate=CLASSIFIER_LR)
classifier_model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

# Callbacks
checkpoint = callbacks.ModelCheckpoint(
    CLASSIFIER_SAVE_PATH, 
    monitor='val_accuracy', 
    save_best_only=True,
    verbose=1, 
    mode='max'
)

early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    verbose=1,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# Train
print("\nStarting training...\n")
classifier_history = classifier_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=CLASSIFIER_EPOCHS,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VAL_STEPS,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

print("\n✓ Classifier training complete!")

### Visualize Classifier Training

In [ ]:
# Plot training history
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axs[0].plot(classifier_history.history['loss'], 'r-', label='Train Loss')
axs[0].plot(classifier_history.history['val_loss'], 'b-', label='Val Loss')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].set_title('Classifier Training Loss')
axs[0].legend()
axs[0].grid(True)

# Accuracy
axs[1].plot(classifier_history.history['accuracy'], 'r-', label='Train Accuracy')
axs[1].plot(classifier_history.history['val_accuracy'], 'b-', label='Val Accuracy')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Accuracy')
axs[1].set_title('Classifier Training Accuracy')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Print best metrics
best_val_acc = max(classifier_history.history['val_accuracy'])
best_epoch = classifier_history.history['val_accuracy'].index(best_val_acc) + 1
print(f"\nBest validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

## 5. Train Segmentation Model

This will train the **Attention U-Net++** segmentation model with **Focal Loss**.

In [ ]:
print("\n" + "="*60)
print("TRAINING SEGMENTATION (Attention U-Net++)")
print("="*60 + "\n")

# Load data
overall_df = pd.read_csv(SEGMENTATION_CSV_PATH, index_col='ImageId')
overall_df_size = len(overall_df)
train_num = int(overall_df_size * TRAIN_PROP)

print(f'Training samples: {train_num}')
print(f'Validation samples: {overall_df_size - train_num}\n')

TRAIN_STEPS = train_num // BATCH_SIZE
VAL_STEPS = (overall_df_size - train_num) // BATCH_SIZE

# Create data generators
train_df = overall_df[:train_num]
val_df = overall_df[train_num:]

train_generator = SegGenerator(train_df, IMAGE_PATH, BATCH_SIZE, resize_to=RESIZE_TO)
val_generator = SegGenerator(val_df, IMAGE_PATH, BATCH_SIZE, resize_to=RESIZE_TO, aug=False)

# Create model
print(f"Creating {SEG_ARCHITECTURE} model (L={SEG_DEPTH}, Attention={USE_ATTENTION})...")
seg_model = create_segmentation_model(
    RESIZE_TO, 
    architecture=SEG_ARCHITECTURE, 
    l=SEG_DEPTH,
    use_attention=USE_ATTENTION
)

# Choose loss function
if LOSS_TYPE == 'combined_dice_wpce':
    seg_loss = combined_dice_wpce_loss(BETA_PIXEL_WEIGHTING, BATCH_SIZE)
    print("Using loss: Combined Dice + Weighted Pixel BCE")
elif LOSS_TYPE == 'combined_dice_focal':
    seg_loss = combined_dice_focal_loss(alpha=0.25, gamma=2.0, dice_weight=2.0, focal_weight=1.0)
    print("Using loss: Combined Dice + Focal Loss (RECOMMENDED)")
elif LOSS_TYPE == 'focal_tversky':
    seg_loss = focal_tversky_loss(alpha=0.7, gamma=0.75)
    print("Using loss: Focal Tversky Loss")
else:
    raise ValueError(f"Unknown loss type: {LOSS_TYPE}")

dice_coefficient = dice_coefficient_wrapper()

# Compile model
opt = optimizers.Adam(learning_rate=SEG_LR)
seg_model.compile(optimizer=opt, loss=seg_loss, metrics=[dice_coefficient])

# Callbacks
checkpoint = callbacks.ModelCheckpoint(
    SEG_SAVE_PATH, 
    monitor='val_dice_coefficient', 
    save_best_only=True,
    verbose=1, 
    mode='max'
)

early_stop = callbacks.EarlyStopping(
    monitor='val_dice_coefficient',
    patience=10,
    verbose=1,
    restore_best_weights=True,
    mode='max'
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Train
print("\nStarting training...\n")
seg_history = seg_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=SEG_EPOCHS,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VAL_STEPS,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

print("\n✓ Segmentation training complete!")

### Visualize Segmentation Training

In [ ]:
# Plot training history
fig, axs = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axs[0].plot(seg_history.history['loss'], 'r-', label='Train Loss')
axs[0].plot(seg_history.history['val_loss'], 'b-', label='Val Loss')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].set_title('Segmentation Training Loss')
axs[0].legend()
axs[0].grid(True)

# Dice Coefficient
axs[1].plot(seg_history.history['dice_coefficient'], 'g-', label='Train Dice')
axs[1].plot(seg_history.history['val_dice_coefficient'], 'm-', label='Val Dice')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Dice Coefficient')
axs[1].set_title('Segmentation Training Dice Score')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Print best metrics
best_val_dice = max(seg_history.history['val_dice_coefficient'])
best_epoch = seg_history.history['val_dice_coefficient'].index(best_val_dice) + 1
print(f"\nBest validation Dice coefficient: {best_val_dice:.4f} at epoch {best_epoch}")

## 6. Training Summary

Run this cell to see a summary of your training results.

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)

print("\n📊 Classifier (EfficientNet-B3):")
print(f"   - Best Val Accuracy: {max(classifier_history.history['val_accuracy']):.4f}")
print(f"   - Final Train Accuracy: {classifier_history.history['accuracy'][-1]:.4f}")
print(f"   - Model saved to: {CLASSIFIER_SAVE_PATH}")

print("\n🎯 Segmentation (Attention U-Net++):")
print(f"   - Best Val Dice: {max(seg_history.history['val_dice_coefficient']):.4f}")
print(f"   - Final Train Dice: {seg_history.history['dice_coefficient'][-1]:.4f}")
print(f"   - Model saved to: {SEG_SAVE_PATH}")

print("\n⚙️ Configuration:")
print(f"   - Classifier Threshold: {CLASSIFIER_THRESHOLD} (lower = higher recall)")
print(f"   - Architecture: {SEG_ARCHITECTURE}")
print(f"   - Loss Function: {LOSS_TYPE}")
print(f"   - Use TTA for inference: Recommended (use_tta=True in predict)")

print("\n✅ Next Steps:")
print("   1. Test predictions using predict.py with use_tta=True")
print("   2. Evaluate performance using performance.py")
print("   3. Adjust classifier_threshold (0.2-0.4) to optimize recall/precision")
print("   4. Use the Streamlit web app for easy deployment")
print("\n" + "="*60)

## 7. Quick Prediction Test (Optional)

Test your trained models on a random validation image.

In [ ]:
from predict import PneumothoraxPredictor

# Create predictor with new models
pp = PneumothoraxPredictor(
    fpath=IMAGE_PATH,
    csv_path=SEGMENTATION_CSV_PATH,
    classifier_path=CLASSIFIER_SAVE_PATH,
    seg_path=SEG_SAVE_PATH,
    classifier_threshold=CLASSIFIER_THRESHOLD
)

# Test prediction with TTA (Test-Time Augmentation)
print("\nTesting prediction with TTA (recommended for better accuracy)...")
pp.predict(use_tta=True)

# You can also test without TTA for faster inference:
# pp.predict(use_tta=False)

## 8. Learning Rate Finder (Optional)

If you want to find the optimal learning rate before training, use this section.

In [ ]:
# Uncomment to run learning rate finder for segmentation model

# from callbacks import LearningRateFinder

# # Create a fresh model
# lrf_model = create_segmentation_model(RESIZE_TO, architecture=SEG_ARCHITECTURE, l=SEG_DEPTH, use_attention=USE_ATTENTION)
# seg_loss = combined_dice_focal_loss()
# dice_coefficient = dice_coefficient_wrapper()
# opt = optimizers.Adam(learning_rate=1e-4)
# lrf_model.compile(optimizer=opt, loss=seg_loss, metrics=[dice_coefficient])

# # Run LRF for 1000 steps
# lrf = LearningRateFinder(1000)
# lrf_model.fit(
#     train_generator,
#     epochs=1,
#     steps_per_epoch=1000,
#     callbacks=[lrf]
# )

# # Plot results
# lrf.plot()

## 9. Model Export Info

Your models are saved in TensorFlow SavedModel format and can be loaded as follows:

In [ ]:
# Example: How to load your saved models later
from tensorflow.keras import models

# Load classifier (no custom objects needed)
# loaded_classifier = models.load_model(CLASSIFIER_SAVE_PATH)

# Load segmentation model (requires custom objects)
# from losses import combined_dice_focal_loss
# from metrics import dice_coefficient_wrapper

# seg_loss = combined_dice_focal_loss()
# dice_coef = dice_coefficient_wrapper()

# loaded_seg_model = models.load_model(
#     SEG_SAVE_PATH,
#     custom_objects={'compute_loss': seg_loss, 'dice_coefficient': dice_coef}
# )

print("Model loading code examples above (commented out)")